# 07e — Offline Semantic Distance + Mobility Alignment

Uses the **full Stage-07d historical OSM cache only**. No ohsome API calls are made.

Questions:

1. How close is each recurring non-HOME anchor to mapped historical semantic features?
2. Does the exact Stage-05b stable-secondary anchor look different from other recurring non-HOME anchors of the same user?
3. How does mapped context overlap with Stage-07b mobility axes?

Historical OSM remains mapping/context evidence, not WORK/OFFICE/HOME ground truth.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='eda/07e-semantic-distance-alignment'
REPO_DIR=Path('/root/geolife')

if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())


In [ ]:
from importlib.util import spec_from_file_location,module_from_spec
import pandas as pd

def load(name,path):
    spec=spec_from_file_location(name,path); m=module_from_spec(spec); sys.modules[name]=m; spec.loader.exec_module(m); return m

s07e=load('s07e',REPO_DIR/'analysis'/'07e_semantic_distance_alignment.py')
print(s07e.synthetic_self_check())

V=Path('/mnt/geolife-data')
C07D=V/'cache'/'07d_historical_context_enrichment'
C=V/'cache'/'07e_semantic_distance_alignment'; C.mkdir(parents=True,exist_ok=True)

def find_required(name, preferred=None):
    if preferred is not None and Path(preferred).exists(): return Path(preferred)
    xs=sorted(V.glob(f'**/{name}'))
    if not xs: raise FileNotFoundError(name)
    return xs[0]

HIST=find_required('historical_context_private.pkl', C07D/'historical_context_private.pkl')
REQLOG=find_required('ohsome_request_log_private.pkl', C07D/'ohsome_request_log_private.pkl')
WORK=find_required('adaptive_work_patterns_42_private.pkl')
PROFILE=find_required('factorized_work_profiles_private.pkl')
RAW=C07D/'ohsome_raw'
if not RAW.exists(): raise FileNotFoundError(RAW)

anchors=pd.read_pickle(HIST)
request_log=pd.read_pickle(REQLOG)
work=pd.read_pickle(WORK)
profiles=pd.read_pickle(PROFILE)
print('anchors:',len(anchors),'users:',anchors['user_id'].astype(str).nunique())
print('successful request-log rows:',request_log['status'].isin(['cached','fetched']).sum())
print('raw parquet files:',len(list(RAW.glob('*.parquet'))))


## 1. Validate full cache and compute geometry distances

Distance is anchor → historical OSM WKB geometry using a local metre transform around each anchor. `bbox` is fallback only if WKB cannot be decoded.

Stage 07d queried an axis-aligned ±~100 m bbox, so 07e uses **exact circular 25/50/100 m thresholds**. Features in bbox corners can be farther than 100 m and are not counted as within-100 m evidence.


In [ ]:
cache_map=s07e.validate_full_ohsome_cache(anchors,request_log,RAW)
assert len(cache_map)==len(anchors)
feature_dist=s07e.build_feature_distance_table(anchors,request_log,RAW)
anchor_metrics=s07e.build_anchor_semantic_metrics(anchors,feature_dist)
print('classified feature-category rows:',len(feature_dist))
print('anchors with semantic feature <=100m:',int(anchor_metrics['semantic_within_100m'].sum()))
print('anchors with work-compatible feature <=100m:',int(anchor_metrics['work_compatible_within_100m'].sum()))


## 2. Align exact stable-secondary anchor and Stage-07b axes

`stable_secondary_anchor` is not inferred here. It is the Stage-05b `dominant_location_id` only when the user's measured `window_pattern == stable_secondary_anchor`.


In [ ]:
aligned=s07e.align_mobility_roles(anchor_metrics,work,profiles)
comparisons=s07e.build_stable_secondary_user_comparisons(aligned)
print('stable-secondary users represented:',aligned.loc[aligned['stable_secondary_user'],'user_id'].nunique())
print('exact stable-secondary anchors:',int(aligned['stable_secondary_anchor'].sum()))
print('within-user comparison rows:',len(comparisons))


In [ ]:
print('WORK-COMPATIBLE DISTANCE BUCKETS')
display(s07e.summarize_distance_buckets(aligned))

print('CATEGORY × DISTANCE THRESHOLD')
category_summary=s07e.summarize_category_thresholds(aligned)
display(category_summary.pivot(index='category',columns='threshold_m',values='anchors').fillna(0).astype(int))

print('STABLE SECONDARY — WITHIN-USER COMPARISON')
stable_summary=s07e.summarize_stable_secondary_comparisons(comparisons)
display(stable_summary)

print('STAGE-07b AXES × MAPPED WORK-COMPATIBLE CONTEXT <=100m')
axis_summary=s07e.summarize_profile_axis_context(aligned)
display(axis_summary)


## 3. Save private + aggregate outputs

Private artifacts retain user/location identifiers and feature-level distances. Aggregate CSVs contain no coordinates or raw OSM IDs.


In [ ]:
feature_dist.to_pickle(C/'feature_semantic_distances_private.pkl')
anchor_metrics.to_pickle(C/'anchor_semantic_distances_private.pkl')
aligned.to_pickle(C/'mobility_aligned_semantic_context_private.pkl')
comparisons.to_pickle(C/'stable_secondary_user_comparisons_private.pkl')

s07e.summarize_distance_buckets(aligned).to_csv(C/'work_context_distance_buckets.csv',index=False)
category_summary.to_csv(C/'category_threshold_summary.csv',index=False)
stable_summary.to_csv(C/'stable_secondary_within_user_summary.csv',index=False)
axis_summary.to_csv(C/'profile_axis_context_summary.csv',index=False)
print('saved:',C)


## Interpretation boundary

- `work-compatible` means a mapped historical OSM category compatible with places where work could occur; it is **not** a WORK/OFFICE label.
- no mapped feature within 100 m means no matching historical OSM feature inside the measured threshold, not real-world absence.
- bootstrap intervals summarize the tiny stable-secondary user cohort; they are not calibrated semantic accuracy intervals.
- Stage 07e makes no occupation inference.
